Model Training and Evaluation: Adversarial Phishing DetectionThis notebook is dedicated to training and evaluating the final Machine Learning (ML) and Deep Learning (DL) models for the phishing detection system. We separate this step from feature engineering to maintain a clean, modular, and reproducible workflow.1. Data Preparation and SplittingThe primary goal of this step is to load the dataset containing all the engineered features (created in the project_overview.ipynb and feature_engineer.py) and split it into training and testing sets.The input data is assumed to be stored as ../processed/sessions_engineered.csv.1.1 Loading DataWe load the data, define the features X and the target label X, and then perform an 80/20 train-test split, ensuring the split is stratified to maintain the original class distribution in both sets.

MLP Deep learning model

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
import random
import joblib

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

DATA_PATH = "../../processed/Feature.csv"  
df = pd.read_csv(DATA_PATH)
df = df.dropna(axis=0)

label_candidates = ['label', 'Label', 'target', 'Target', 'y']
label_col = next((c for c in label_candidates if c in df.columns), df.columns[-1])

X = df.drop(columns=[label_col])
y = df[label_col]

if y.dtype == object or not np.issubdtype(y.dtype, np.number):
    le = LabelEncoder()
    y = le.fit_transform(y)

numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
])

X_processed = preprocessor.fit_transform(X)


kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
accuracies, precisions, recalls, f1s = [], [], [], []

for train_idx, test_idx in kf.split(X_processed, y):
    X_train, X_test = X_processed[train_idx], X_processed[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    
    classes = np.unique(y_train)
    class_weights_values = compute_class_weight('balanced', classes=classes, y=y_train)
    class_weights = dict(zip(classes, class_weights_values))

    
    input_dim = X_train.shape[1]
    model = Sequential([
        Dense(32, input_dim=input_dim, activation='relu', kernel_regularizer=l2(0.001)),
        BatchNormalization(),
        Dropout(0.5),
        Dense(16, activation='relu', kernel_regularizer=l2(0.001)),
        Dropout(0.4),
        Dense(1, activation='sigmoid')
    ])

    model.compile(optimizer=Adam(learning_rate=0.001),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])

    
    early_stop = EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)

    
    model.fit(X_train, y_train,
              validation_split=0.2,
              epochs=50,
              batch_size=16,
              verbose=0,
              callbacks=[early_stop],
              class_weight=class_weights)

    
    y_pred = (model.predict(X_test) > 0.5).astype(int)
    accuracies.append(accuracy_score(y_test, y_pred))
    precisions.append(precision_score(y_test, y_pred))
    recalls.append(recall_score(y_test, y_pred))
    f1s.append(f1_score(y_test, y_pred))


print("=== Optimized Stable MLP Metrics (5-Fold CV) ===")
print(f"Accuracy: {np.mean(accuracies):.4f}")
print(f"Precision: {np.mean(precisions):.4f}")
print(f"Recall: {np.mean(recalls):.4f}")
print(f"F1-score: {np.mean(f1s):.4f}")
model.save("mlp_stable_model.keras")  



joblib.dump(preprocessor, "mlp_preprocessor.pkl")

print("\nModel and preprocessor saved successfully!")
model = load_model("mlp_stable_model.keras")
preprocessor = joblib.load("mlp_preprocessor.pkl")


TEST_PATH = "../../processed/test_mlp.csv"  
test_df = pd.read_csv(TEST_PATH)

if 'label' in test_df.columns:
    y_test = test_df['label']
    X_test = test_df.drop(columns=['label'])
else:
    y_test = None
    X_test = test_df.copy()


train_cols = X.columns.tolist()

missing = [c for c in train_cols if c not in X_test.columns]
for c in missing:
    if c in numeric_cols:
        X_test[c] = 0
    else:
        X_test[c] = ""  


extra = [c for c in X_test.columns if c not in train_cols]
if extra:
    X_test = X_test.drop(columns=extra)


X_test = X_test[train_cols]

X_test_processed = preprocessor.transform(X_test)


y_pred = (model.predict(X_test_processed) > 0.5).astype(int)


if y_test is not None:
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    print("\n=== Test Data Metrics ===")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-score: {f1:.4f}")
else:
    print("\nPredictions on test data:")
    print(y_pred)


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

def load_and_prepare_data():
    """Load and prepare the feature data for Random Forest model training"""
    # Load the combined dataset
    df = pd.read_csv('D:/python project(cse466)/Fake-Payment-Gateway-Detection-Adversarial-Resistant-/Data/Processed/combined_dataset.csv')
    
    # Feature engineering
    df['url_length'] = df['url'].apply(len)
    df['num_dots'] = df['url'].apply(lambda x: x.count('.'))
    df['has_https'] = df['url'].apply(lambda x: 1 if x.startswith('https') else 0)
    df['num_digits'] = df['url'].apply(lambda x: sum(c.isdigit() for c in x))
    df['num_special_chars'] = df['url'].apply(lambda x: sum(not c.isalnum() for c in x))
    df['has_ip'] = df['url'].apply(lambda x: 1 if any(part.isdigit() for part in x.split('.')) else 0)
    
    # Encode categorical variables
    le = LabelEncoder()
    df['Location_encoded'] = le.fit_transform(df['Location'])
    
    # Select features for modeling
    feature_columns = ['TransactionAmount', 'CustomerAge', 'AccountBalance', 
                      'url_length', 'num_dots', 'has_https', 'num_digits', 
                      'num_special_chars', 'has_ip', 'Location_encoded']
    
    X = df[feature_columns]
    y = df['label']
    
    # Handle missing values
    X = X.fillna(0)
    
    return X, y, df, feature_columns

def train_random_forest_model(X, y):
    """Train a Random Forest model with optimized parameters"""
    # Split the data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
    
    # Create Random Forest model with optimized parameters
    rf_model = RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1  # Use all processors
    )
    
    # Train the model
    rf_model.fit(X_train, y_train)
    
    # Make predictions
    y_pred = rf_model.predict(X_test)
    y_pred_proba = rf_model.predict_proba(X_test)
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    return rf_model, X_test, y_test, y_pred, y_pred_proba, accuracy, precision, recall, f1

def display_feature_importance(model, feature_names):
    """Display feature importance from the Random Forest model"""
    importances = model.feature_importances_
    indices = np.argsort(importances)[::-1]
    
    print("\nFeature Importance Rankings:")
    print("-" * 40)
    for i in range(len(feature_names)):
        print(f"{i+1}. {feature_names[indices[i]]}: {importances[indices[i]]:.4f}")

def evaluate_model_performance(accuracy, precision, recall, f1):
    """Display model performance metrics"""
    print("\n" + "="*50)
    print("RANDOM FOREST MODEL PERFORMANCE")
    print("="*50)
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-Score:  {f1:.4f}")

def main():
    """Main function to run the Random Forest implementation"""
    print("Fake Payment Gateway Detection - Random Forest Implementation")
    print("=" * 65)
    
    try:
        # Load and prepare data
        X, y, df, feature_columns = load_and_prepare_data()
        print(f"Dataset shape: {df.shape}")
        print("Label distribution:")
        print(df['label'].value_counts())
        
        # Train Random Forest model
        model, X_test, y_test, y_pred, y_pred_proba, accuracy, precision, recall, f1 = train_random_forest_model(X, y)
        
        # Evaluate model performance
        evaluate_model_performance(accuracy, precision, recall, f1)
        
        # Display classification report
        print("\nDetailed Classification Report:")
        print("-" * 40)
        print(classification_report(y_test, y_pred, target_names=['Legitimate', 'Phishing']))
        
        # Display feature importance
        display_feature_importance(model, feature_columns)
        
        # Save the trained model
        import joblib
        joblib.dump(model, 'D:/python project(cse466)/Fake-Payment-Gateway-Detection-Adversarial-Resistant-/Data/Notebook/Models/random_forest_model.pkl')
        print("\nModel saved successfully as 'random_forest_model.pkl'")
        
    except Exception as e:
        print(f"An error occurred: {str(e)}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

Fake Payment Gateway Detection - Random Forest Implementation
Dataset shape: (100, 13)
Label distribution:
label
1    50
0    50
Name: count, dtype: int64

RANDOM FOREST MODEL PERFORMANCE
Accuracy:  1.0000
Precision: 1.0000
Recall:    1.0000
F1-Score:  1.0000

Detailed Classification Report:
----------------------------------------
              precision    recall  f1-score   support

  Legitimate       1.00      1.00      1.00        15
    Phishing       1.00      1.00      1.00        15

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30


Feature Importance Rankings:
----------------------------------------
1. num_digits: 0.5969
2. url_length: 0.1883
3. AccountBalance: 0.1062
4. TransactionAmount: 0.0399
5. CustomerAge: 0.0269
6. num_special_chars: 0.0159
7. Location_encoded: 0.0157
8. num_dots: 0.0102
9. has_ip: 0.0000
10. has_https: 0.0000

Model saved successfully as 